# NHANES 2017-2018 Depression Analysis

This notebook loads NHANES 2017-2018 modules, merges them by participant ID, creates a binary depression label from PHQ-9, plots class distribution, and summarizes missing values.

In [ ]:
# Imports
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import seaborn as sns

warnings.filterwarnings('ignore')

print("All libraries loaded successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# Load NHANES 2017-2018 modules
# pyreadstat 1.2.x reads SAS transport files with read_xport.
DATA_DIR = Path('../data/nhanes')

demo, _ = pyreadstat.read_xport(DATA_DIR / 'DEMO_J.XPT')
dpq, _ = pyreadstat.read_xport(DATA_DIR / 'DPQ_J.XPT')
paq, _ = pyreadstat.read_xport(DATA_DIR / 'PAQ_J.XPT')
hoq, _ = pyreadstat.read_xport(DATA_DIR / 'HOQ_J.XPT')

print(f"DEMO shape: {demo.shape}")
print(f"DPQ  shape: {dpq.shape}")
print(f"PAQ  shape: {paq.shape}")
print(f"HOQ  shape: {hoq.shape}")

In [ ]:
# Merge all modules on SEQN
df = demo.merge(dpq, on='SEQN', how='inner') \
         .merge(paq, on='SEQN', how='inner') \
         .merge(hoq, on='SEQN', how='inner')

print(f"Merged dataset shape: {df.shape}")
df.head()

In [ ]:
# Create depression label from PHQ-9
# PHQ-9 total = sum of items DPQ010 to DPQ090
phq_cols = [
    'DPQ010', 'DPQ020', 'DPQ030', 'DPQ040', 'DPQ050',
    'DPQ060', 'DPQ070', 'DPQ080', 'DPQ090'
]

df['PHQ9_TOTAL'] = df[phq_cols].apply(
    pd.to_numeric, errors='coerce'
).sum(axis=1, min_count=9)

# Binary label: 1 = depressed (PHQ9 >= 10), 0 = not depressed
df['DEPRESSION'] = (df['PHQ9_TOTAL'] >= 10).astype(int)

print(df['DEPRESSION'].value_counts())
print(f"\nDepression prevalence: {df['DEPRESSION'].mean()*100:.1f}%")
print(f"Missing PHQ9_TOTAL: {df['PHQ9_TOTAL'].isna().sum()}")

**Note:** Rows with missing `PHQ9_TOTAL` are labeled as `0` by the binary comparison above. For reporting prevalence, exclude missing PHQ-9 totals unless the methodology decides otherwise.

In [ ]:
# Depression prevalence among rows with complete PHQ-9 responses
valid_phq = df.dropna(subset=['PHQ9_TOTAL']).copy()
valid_phq['DEPRESSION'] = (valid_phq['PHQ9_TOTAL'] >= 10).astype(int)

print(valid_phq['DEPRESSION'].value_counts())
print(f"\nValid PHQ-9 rows: {len(valid_phq)}")
print(f"Depression prevalence excluding missing PHQ-9: {valid_phq['DEPRESSION'].mean()*100:.1f}%")

In [ ]:
# Class distribution plot
Path('../results').mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(7, 4))
ax = sns.countplot(
    x='DEPRESSION',
    data=df,
    hue='DEPRESSION',
    palette=['#2ecc71', '#e74c3c'],
    legend=False,
)
plt.title('Class Distribution - Depression Label (PHQ-9 >= 10)', fontsize=13, fontweight='bold')
plt.xlabel('Depression Label (0 = Not Depressed, 1 = Depressed)')
plt.ylabel('Count')

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha='center',
        va='bottom',
        fontsize=11,
    )

plt.tight_layout()
plt.savefig('../results/class_distribution.png', dpi=150)
plt.show()
print('Plot saved to results/')

In [ ]:
# Missing value summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct,
}).sort_values('Missing %', ascending=False)

print('Top 20 features with missing values:')
print(missing_df[missing_df['Missing Count'] > 0].head(20))